In [13]:
import pandas as pd
import numpy as np 

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import xgboost as xgb
import joblib
import mlflow
import mlflow.sklearn

In [14]:
#Cargar datos
df = pd.read_csv('../data/processed/train_clean.csv')
X = df.drop('SalePrice', axis= 1) #Matrix
y = df['SalePrice'] #Vector

In [15]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [16]:
X_train.shape, y_train.shape

((1168, 247), (1168,))

In [17]:
#Instancia del modelo
model = LinearRegression()
model.fit(X_train, y_train)
predictions = model.predict(X_test)

In [18]:
y_pred = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))


In [19]:
print(np.exp(rmse))
print(f"RMSE en dólares aprox: ${np.exp(rmse):,.0f}")

1.2346033733317907
RMSE en dólares aprox: $1


In [20]:
joblib.dump(model, '../models/linear_regression.pkl')

['../models/linear_regression.pkl']

In [21]:
mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("property-valuation")

with mlflow.start_run():
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("max_depth", 6)
    
    model_xgbo = xgb.XGBRegressor(n_estimators=200, max_depth=6)
    model_xgbo.fit(X_train, y_train)
    
    y_pred = model_xgbo.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mlflow.log_metric("rmse", rmse)
    print(f"RMSE: {rmse:.4f}")



RMSE: 0.1422


In [22]:
print(np.exp(rmse))
print(f"RMSE en dólares aprox: ${np.exp(rmse):,.0f}")

1.1528048685012313
RMSE en dólares aprox: $1


In [23]:

joblib.dump(model_xgbo, '../models/xgboost_model.pkl')

['../models/xgboost_model.pkl']

In [24]:
import json

columns = X_train.columns.tolist()
with open('../models/columns.json', 'w') as f:
    json.dump(columns, f)